In [28]:
# Paste in Jupyter, paste output back
from pymongo import MongoClient
import os

uri = os.environ.get("MONGO_URI", "mongodb://localhost:27017")
client = MongoClient(uri)
db = client[os.environ.get("MONGO_DATABASE", "quants_lab")]

pipeline = [
    {"$match": {"trading_pair": "XMR-USDT", "connector": {"$in": ["mexc", "nonkyc"]}}},
    {"$group": {
        "_id": {"connector": "$connector", "interval": "$interval"},
        "count": {"$sum": 1},
        "first_ts": {"$min": "$timestamp"},
        "last_ts":  {"$max": "$timestamp"},
    }},
    {"$sort": {"_id.connector": 1, "_id.interval": 1}},
]
for doc in db["candles"].aggregate(pipeline):
    c = doc["_id"]["connector"]
    i = doc["_id"]["interval"]
    n = doc["count"]
    span_days = (doc["last_ts"] - doc["first_ts"]) / 86400
    print(f"{c:>8}  {i:>4}  {n:>7} bars   {span_days:6.1f} days")

    mexc   15m    34599 bars    360.4 days
    mexc    1d     2318 bars   2317.0 days
    mexc    1h     5410 bars    225.4 days
    mexc    1m   108748 bars     76.2 days
    mexc    4h     1351 bars    225.0 days
    mexc    5m   103802 bars    361.0 days
    mexc    8h      540 bars    179.7 days
  nonkyc   12h      360 bars    179.5 days
  nonkyc   15m    96889 bars   1053.5 days
  nonkyc    1d     1079 bars   1079.0 days
  nonkyc    1h    25541 bars   1079.5 days
  nonkyc    4h     6446 bars   1079.3 days
  nonkyc    5m   201414 bars    731.1 days
  nonkyc    8h      540 bars    179.7 days


In [1]:
# MarketLab Phase 2 / Phase 3 diagnostic v3
# IMPORTANT: RESTART YOUR JUPYTER KERNEL BEFORE RUNNING THIS to clear zombies
# from v2's failed spawn pool.
# Expected run time: 5-7 minutes.

import os, sys, time, pickle, gc
import numpy as np
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

print("=" * 80)
print("SECTION 0 — Environment")
print("=" * 80)
print(f"Python: {sys.version.split()[0]}")
print(f"Default mp start method: {mp.get_start_method()}")
print(f"Available start methods: {mp.get_all_start_methods()}")
print(f"CPU count: {os.cpu_count()}")

print()
print("=" * 80)
print("SECTION 1 — Baseline import cost")
print("=" * 80)

t0 = time.perf_counter()
from pmm_lab.objective.signal_cache import SharedSignalCache, signal_cache_key
from pmm_lab.strategies.mean_reversion_bb_rsi import MeanReversionBBRSIStrategyConfig
from pmm_lab.strategies.ema_regime_hold import EMARegimeHoldStrategyConfig
from pmm_lab.config.params import PairRules, FeeConfig
print(f"pmm_lab import cost (warm): {time.perf_counter()-t0:.3f}s")

print()
print("=" * 80)
print("SECTION 2 — Build synthetic datasets")
print("=" * 80)

# Big dataset for Section 3 (single-config feature kernel timing — we already
# know this is the key metric, we want a realistic measurement).
BARS_FULL = 16000  # ~55 days of 5m bars

# Smaller dataset for sections 6-7 (which do MANY controller-compat calls).
# The ratio between selection and precompute time stays meaningful at smaller
# scales, and this keeps total diagnostic runtime under ~6 minutes.
BARS_SMALL = 6000  # ~21 days of 5m bars

def make_candles(n_bars: int, seed: int = 42):
    rng = np.random.default_rng(seed)
    dtype = np.dtype([
        ("timestamp", "i8"),
        ("open", "f8"), ("high", "f8"), ("low", "f8"), ("close", "f8"),
        ("volume", "f8"),
    ])
    c = np.zeros(n_bars, dtype=dtype)
    c["timestamp"] = np.arange(n_bars) * 300
    price = 100.0 + np.cumsum(rng.standard_normal(n_bars) * 0.05)
    c["close"] = price
    c["open"] = price + rng.standard_normal(n_bars) * 0.02
    c["high"] = np.maximum(c["open"], c["close"]) + np.abs(rng.standard_normal(n_bars)) * 0.03
    c["low"]  = np.minimum(c["open"], c["close"]) - np.abs(rng.standard_normal(n_bars)) * 0.03
    c["volume"] = 1000 + np.abs(rng.standard_normal(n_bars)) * 200
    return c

candles_full = make_candles(BARS_FULL)
regime_full = candles_full[::12].copy()
candles_small = make_candles(BARS_SMALL)
regime_small = candles_small[::12].copy()
print(f"Full candles: {len(candles_full)} bars ({candles_full.nbytes/1024:.1f} KiB)")
print(f"Small candles: {len(candles_small)} bars ({candles_small.nbytes/1024:.1f} KiB)")

pair_rules = PairRules(
    price_tick=0.01,
    amount_step=0.00001,
    min_notional_quote=5.0,
    min_order_size_base=0.00001,
    fees=FeeConfig(maker_fee=0.001, taker_fee=0.001),
    supports_post_only=True,
)

print()
print("=" * 80)
print("SECTION 3 — Controller-compat feature-kernel cost (already measured in v2)")
print("=" * 80)
print("We already know these numbers from v2. Re-measuring briefly as a sanity check.")

mr_cfg_cc = MeanReversionBBRSIStrategyConfig(controller_compat=True)
ema_cfg_cc = EMARegimeHoldStrategyConfig(controller_compat=True)
mr_cfg_vec = MeanReversionBBRSIStrategyConfig(controller_compat=False)
ema_cfg_vec = EMARegimeHoldStrategyConfig(controller_compat=False)

def time_compute(cfg, n_bars, is_ema=False):
    _c = candles_full if n_bars == BARS_FULL else candles_small
    _r = (regime_full if n_bars == BARS_FULL else regime_small) if is_ema else None
    cache = SharedSignalCache()
    t0 = time.perf_counter()
    cache.get_or_compute(cfg, "dev", _c, pair_rules, regime_candles=_r)
    return time.perf_counter() - t0

# Quick re-measure on small dataset (so this finishes in under 30 seconds)
print(f"(Using BARS_SMALL={BARS_SMALL} for a quick re-check. v2 used {BARS_FULL}.)")
_ = time_compute(mr_cfg_vec, BARS_SMALL)  # warm
mr_vec_t = time_compute(mr_cfg_vec, BARS_SMALL)
mr_cc_t  = time_compute(mr_cfg_cc, BARS_SMALL)
ema_vec_t = time_compute(ema_cfg_vec, BARS_SMALL, is_ema=True)
ema_cc_t  = time_compute(ema_cfg_cc, BARS_SMALL, is_ema=True)
print(f"  MR  vec={mr_vec_t:.3f}s  cc={mr_cc_t:.3f}s  ratio={mr_cc_t/mr_vec_t:.0f}x  per-bar={mr_cc_t*1e6/BARS_SMALL:.0f}us")
print(f"  EMA vec={ema_vec_t:.3f}s  cc={ema_cc_t:.3f}s  ratio={ema_cc_t/ema_vec_t:.0f}x  per-bar={ema_cc_t*1e6/BARS_SMALL:.0f}us")

print()
print("=" * 80)
print("SECTION 4 — Pickle cost (quick check)")
print("=" * 80)
t0 = time.perf_counter()
cb = pickle.dumps(candles_small, protocol=pickle.HIGHEST_PROTOCOL)
t_p = time.perf_counter() - t0
t0 = time.perf_counter()
_ = pickle.loads(cb)
t_u = time.perf_counter() - t0
print(f"  Round-trip small candles: {(t_p+t_u)*1000:.1f}ms ({len(cb)/1024:.1f} KiB)")

print()
print("=" * 80)
print("SECTION 5 — Fork pool startup cost (spawn skipped — Jupyter-incompatible)")
print("=" * 80)

def _noop_worker(_dummy):
    """Takes one arg because pool.map passes items from the iterable."""
    return os.getpid()

if "fork" in mp.get_all_start_methods():
    ctx = mp.get_context("fork")
    try:
        with ProcessPoolExecutor(max_workers=2, mp_context=ctx) as p:
            pids = list(p.map(_noop_worker, range(2)))
        t0 = time.perf_counter()
        with ProcessPoolExecutor(max_workers=2, mp_context=ctx) as p:
            pids = list(p.map(_noop_worker, range(2)))
        t_cold = time.perf_counter() - t0
        print(f"  Fork pool create + 2 noop tasks: {t_cold*1000:.1f}ms  pids={pids}")
    except Exception as e:
        print(f"  Fork pool test failed: {type(e).__name__}: {e}")
else:
    print("  Fork not available on this platform (skipping)")

# Intentionally do NOT test spawn here — defining _noop_worker in a notebook
# cell means spawn-context workers can't unpickle it from __main__. That is a
# Jupyter limitation, not a real issue with your production code (which imports
# workers from real modules).

print()
print("=" * 80)
print("SECTION 6 — Phase 2 precompute: serial vs parallel (6 unique configs)")
print("=" * 80)
print(f"Using BARS_SMALL={BARS_SMALL}. Expect serial precompute ~{6 * mr_cc_t:.0f}s.")

# 6 distinct MR configs by signal-affecting params
mr_candidates = []
for i, (bb_len, rsi_len) in enumerate([(80,14),(96,14),(120,14),(80,10),(80,20),(64,14)]):
    cfg = MeanReversionBBRSIStrategyConfig(
        bb_length=bb_len, rsi_length=rsi_len, controller_compat=True,
    )
    mr_candidates.append({"config": cfg, "trial_number": i, "phase1_score": 1.0 - i*0.01})

keys = {signal_cache_key(c["config"]) for c in mr_candidates}
print(f"  Candidate count: {len(mr_candidates)}, unique signal keys: {len(keys)}")

# Serial baseline
gc.collect()
cache_serial = SharedSignalCache()
t0 = time.perf_counter()
for c in mr_candidates:
    cache_serial.get_or_compute(c["config"], "dev", candles_small, pair_rules)
t_serial = time.perf_counter() - t0
print(f"  SERIAL precompute: {t_serial:7.3f}s  ({t_serial/len(mr_candidates):.3f}s/candidate)")

# Stage 3 parallel path (if installed)
has_stage3 = False
try:
    from pmm_lab.objective.phase2_parallel_directional import precompute_unique_directional_signals
    has_stage3 = True
    print("  Stage 3 module IS installed")
except ImportError as e:
    print(f"  Stage 3 module not installed — {e}")

if has_stage3:
    for n_workers in (2, 4, 6):
        gc.collect()
        try:
            t0 = time.perf_counter()
            _ = precompute_unique_directional_signals(
                top_candidates=mr_candidates,
                candles=candles_small,
                pair_rules=pair_rules,
                regime_candles=None,
                dataset_key="dev",
                max_workers=n_workers,
            )
            t_par = time.perf_counter() - t0
            speedup = t_serial / t_par if t_par > 0 else float("nan")
            verdict = "GOOD" if speedup >= 1.5 else "WEAK" if speedup >= 1.1 else "NO WIN"
            print(f"  PARALLEL {n_workers}w: {t_par:7.3f}s  speedup={speedup:.2f}x  [{verdict}]")
        except Exception as e:
            print(f"  PARALLEL {n_workers}w: FAILED — {type(e).__name__}: {e}")

print()
print("=" * 80)
print("SECTION 7 — Selection-loop vs precompute wall-time ratio (MR)")
print("=" * 80)
print("THE KEY QUESTION. If selection dominates, Stage 3 parallelism can't")
print("help much regardless of worker count.")

try:
    from pmm_lab.objective.stress_selection import select_best_stressed_candidate
    from pmm_lab.objective.stress_mean_reversion_bb_rsi import _apply_scenario as _mr_apply_scenario
    from pmm_lab.optuna.canonicalizer_mean_reversion_bb_rsi import canonicalize_params
    from pmm_lab.objective.stress import load_stress_scenarios
    from dataclasses import replace as _replace

    mr_full = []
    for i, (bb_len, rsi_len) in enumerate([(80,14),(96,14),(120,14),(80,10),(80,20),(64,14)]):
        raw = {
            "bb_length": bb_len, "bb_std": 2.0, "bbp_entry_threshold": 0.20,
            "rsi_length": rsi_len, "rsi_entry_threshold": 40.0,
            "use_trend_filter": True, "trend_ema_length": 200, "min_trend_slope": 0.0,
            "atr_length": 14, "max_atr_pct_for_entry": 0.10,
            "volume_filter_window": 288, "min_volume_quantile": 0.30,
            "max_spread_pct": 0.006, "max_trades_per_day": 6,
            "max_executors_per_side": 1, "total_amount_quote": 300.0,
            "stop_loss": 0.02, "take_profit": 0.03, "cooldown_time": 300.0,
            "time_limit": 3600, "take_profit_order_type": "market",
            "trailing_stop_activation": 0.0, "trailing_stop_delta": 0.0,
        }
        bundle, rej = canonicalize_params(raw, pair_rules, 100.0, bar_interval_seconds=300)
        if bundle is None:
            print(f"  canonicalize failed for candidate {i}: {rej}")
            continue
        sc = _replace(bundle.strategy_config, controller_compat=True)
        mr_full.append({
            "trial_number": i,
            "phase1_score": 1.0 - i*0.01,
            "params": raw,
            "config": sc,
            "engine_config": bundle.engine_config,
        })
    print(f"  Built {len(mr_full)} full candidates (with engine_config)")

    # Precompute signals (reuse cache_serial from Section 6 if keys align)
    gc.collect()
    shared = SharedSignalCache()
    t0 = time.perf_counter()
    for c in mr_full:
        shared.get_or_compute(c["config"], "dev", candles_small, pair_rules)
    t_pre = time.perf_counter() - t0
    print(f"  Serial precompute: {t_pre:.3f}s")

    def _apply_fn(strat, eng, rules, scenario):
        new_eng, new_rules = _mr_apply_scenario(eng, rules, scenario)
        return strat, new_eng, new_rules

    scenarios = load_stress_scenarios()
    print(f"  Stress scenarios loaded: {len(scenarios)}")

    t0 = time.perf_counter()
    best, diag = select_best_stressed_candidate(
        mr_full, candles_small, pair_rules, 300,
        scenarios=scenarios,
        objective_version=2,
        shared_signal_cache=shared,
        dataset_key="dev",
        apply_scenario_fn=_apply_fn,
    )
    t_sel = time.perf_counter() - t0
    print(f"  Selection loop:    {t_sel:.3f}s")

    total = t_pre + t_sel
    sel_pct = 100 * t_sel / total if total > 0 else 0
    pre_pct = 100 * t_pre / total if total > 0 else 0
    print(f"  Ratio: precompute={pre_pct:.0f}%  selection={sel_pct:.0f}%")
    print(f"  Diag: evaluated={diag['candidates_evaluated']} "
          f"pruned={diag['candidates_pruned']} "
          f"hits={diag['signal_cache_hits']} misses={diag['signal_cache_misses']}")

    print()
    print("  >>> INTERPRETATION <<<")
    if sel_pct > 50:
        print("  >>> Selection loop dominates. Stage 3 (parallel precompute) alone")
        print("  >>> has a ceiling. Compiled kernels (Stage 5) would help the")
        print("  >>> selection loop's per-simulation cost too — bigger win.")
    elif sel_pct > 25:
        print("  >>> Precompute and selection roughly balanced. Stage 5 (compiled")
        print("  >>> kernels) wins on BOTH fronts since the selection loop's")
        print("  >>> per-scenario work also uses the same feature kernels.")
    else:
        print("  >>> Precompute dominates. Stage 3 parallelism SHOULD have helped.")
        print("  >>> If it didn't, something is wrong with the Stage 3 integration.")
except Exception as e:
    import traceback
    print(f"  Section 7 failed: {type(e).__name__}: {e}")
    traceback.print_exc()

print()
print("=" * 80)
print("SECTION 8 — Realistic dedup ratio at production TOP_N=100")
print("=" * 80)
print("How many UNIQUE signal keys does production TOP_N=100 actually produce?")
print("This determines the real parallelism ceiling for Stage 3.")

def _count_unique_keys_mr(n):
    rng = np.random.default_rng(1)
    keys = set()
    for _ in range(n):
        cfg = MeanReversionBBRSIStrategyConfig(
            bb_length=int(rng.integers(40, 120)),
            bb_std=float(rng.uniform(1.8, 2.5)),
            bbp_entry_threshold=float(rng.uniform(0.15, 0.25)),
            rsi_length=int(rng.integers(10, 20)),
            rsi_entry_threshold=float(rng.uniform(30, 45)),
            max_atr_pct_for_entry=float(rng.uniform(0.08, 0.12)),
            min_volume_quantile=float(rng.uniform(0.25, 0.35)),
            controller_compat=True,
        )
        keys.add(signal_cache_key(cfg))
    return len(keys)

def _count_unique_keys_ema(n):
    rng = np.random.default_rng(1)
    keys = set()
    for _ in range(n):
        cfg = EMARegimeHoldStrategyConfig(
            regime_ema_fast=int(rng.integers(20, 80)),
            regime_ema_slow=int(rng.integers(100, 300)),
            regime_adx_length=int(rng.integers(10, 20)),
            regime_adx_threshold=float(rng.uniform(15, 25)),
            min_volume_quantile=float(rng.uniform(0.25, 0.35)),
            controller_compat=True,
        )
        keys.add(signal_cache_key(cfg))
    return len(keys)

for n in (15, 75, 100):
    km = _count_unique_keys_mr(n)
    ke = _count_unique_keys_ema(n)
    print(f"  TOP_N={n}: MR={km} unique ({100*km/n:.0f}%)  EMA={ke} unique ({100*ke/n:.0f}%)")

print()
print("  Production TOP_N=100 wall-time projections based on Section 3:")
print(f"  MR serial @ 100 unique keys:  ~{mr_cc_t * BARS_FULL/BARS_SMALL * 100:.0f}s = "
      f"{mr_cc_t * BARS_FULL/BARS_SMALL * 100 / 60:.1f} min per pair")
print(f"  EMA serial @ 100 unique keys: ~{ema_cc_t * BARS_FULL/BARS_SMALL * 100:.0f}s = "
      f"{ema_cc_t * BARS_FULL/BARS_SMALL * 100 / 60:.1f} min per pair")

print()
print("=" * 80)
print("DONE — paste full output back")
print("=" * 80)

SECTION 0 — Environment
Python: 3.12.13
Default mp start method: fork
Available start methods: ['fork', 'spawn', 'forkserver']
CPU count: 32

SECTION 1 — Baseline import cost
pmm_lab import cost (warm): 0.280s

SECTION 2 — Build synthetic datasets
Full candles: 16000 bars (750.0 KiB)
Small candles: 6000 bars (281.2 KiB)

SECTION 3 — Controller-compat feature-kernel cost (already measured in v2)
We already know these numbers from v2. Re-measuring briefly as a sanity check.
(Using BARS_SMALL=6000 for a quick re-check. v2 used 16000.)
  MR  vec=0.005s  cc=13.545s  ratio=2543x  per-bar=2258us
  EMA vec=0.025s  cc=22.092s  ratio=879x  per-bar=3682us

SECTION 4 — Pickle cost (quick check)
  Round-trip small candles: 0.3ms (281.5 KiB)

SECTION 5 — Fork pool startup cost (spawn skipped — Jupyter-incompatible)
  Fork pool create + 2 noop tasks: 11.5ms  pids=[25295, 25296]

SECTION 6 — Phase 2 precompute: serial vs parallel (6 unique configs)
Using BARS_SMALL=6000. Expect serial precompute ~81s.

Traceback (most recent call last):
  File "/tmp/ipykernel_25262/824427826.py", line 213, in <module>
    from pmm_lab.optuna.canonicalizer_mean_reversion_bb_rsi import canonicalize_params
ImportError: cannot import name 'canonicalize_params' from 'pmm_lab.optuna.canonicalizer_mean_reversion_bb_rsi' (/quants-lab/research_notebooks/market_lab/pmm_dynamic/pmm_lab/optuna/canonicalizer_mean_reversion_bb_rsi.py)


In [2]:
# Section 7 fix — paste as a NEW cell in Jupyter after v3 has run.
# Requires: `candles_small`, `pair_rules`, `SharedSignalCache`, etc. from v3 (already loaded).
#
# What this measures: wall time split between Phase 2 signal precompute and
# the selection loop itself. If selection loop dominates, Stage 3 (parallel
# precompute) has a ceiling regardless of worker count.
#
# Expected runtime: ~1.5 minutes.

import gc, time
from dataclasses import replace as _replace

from pmm_lab.objective.stress_selection import select_best_stressed_candidate
from pmm_lab.objective.stress_mean_reversion_bb_rsi import _apply_scenario as _mr_apply_scenario
# Real export name is canonicalize_mr_bb_rsi_params — the notebook aliases it.
from pmm_lab.optuna.canonicalizer_mean_reversion_bb_rsi import canonicalize_mr_bb_rsi_params
from pmm_lab.objective.stress import load_stress_scenarios

print("=" * 80)
print("SECTION 7 (retry) — MR Selection loop vs precompute wall-time ratio")
print("=" * 80)

mr_full = []
for i, (bb_len, rsi_len) in enumerate([(80,14),(96,14),(120,14),(80,10),(80,20),(64,14)]):
    raw = {
        "bb_length": bb_len, "bb_std": 2.0, "bbp_entry_threshold": 0.20,
        "rsi_length": rsi_len, "rsi_entry_threshold": 40.0,
        "use_trend_filter": True, "trend_ema_length": 200, "min_trend_slope": 0.0,
        "atr_length": 14, "max_atr_pct_for_entry": 0.10,
        "volume_filter_window": 288, "min_volume_quantile": 0.30,
        "max_spread_pct": 0.006, "max_trades_per_day": 6,
        "max_executors_per_side": 1, "total_amount_quote": 300.0,
        "stop_loss": 0.02, "take_profit": 0.03, "cooldown_time": 300.0,
        "time_limit": 3600, "take_profit_order_type": "market",
        "trailing_stop_activation": 0.0, "trailing_stop_delta": 0.0,
    }
    bundle, rej = canonicalize_mr_bb_rsi_params(raw, pair_rules, 100.0, bar_interval_seconds=300)
    if bundle is None:
        print(f"  canonicalize failed for candidate {i}: {rej}")
        continue
    sc = _replace(bundle.strategy_config, controller_compat=True)
    mr_full.append({
        "trial_number": i,
        "phase1_score": 1.0 - i*0.01,
        "params": raw,
        "config": sc,
        "engine_config": bundle.engine_config,
    })
print(f"  Built {len(mr_full)} full candidates (with engine_config)")

# Precompute signals (serial, so we measure the two phases cleanly)
gc.collect()
shared = SharedSignalCache()
t0 = time.perf_counter()
for c in mr_full:
    shared.get_or_compute(c["config"], "dev", candles_small, pair_rules)
t_pre = time.perf_counter() - t0
print(f"  Serial precompute: {t_pre:.3f}s  ({t_pre/len(mr_full):.3f}s per candidate)")

def _apply_fn(strat, eng, rules, scenario):
    new_eng, new_rules = _mr_apply_scenario(eng, rules, scenario)
    return strat, new_eng, new_rules

scenarios = load_stress_scenarios()
print(f"  Stress scenarios loaded: {len(scenarios)}")

# Selection loop (signals already cached — this times JUST the selection work)
t0 = time.perf_counter()
best, diag = select_best_stressed_candidate(
    mr_full, candles_small, pair_rules, 300,
    scenarios=scenarios,
    objective_version=2,
    shared_signal_cache=shared,
    dataset_key="dev",
    apply_scenario_fn=_apply_fn,
)
t_sel = time.perf_counter() - t0
print(f"  Selection loop:    {t_sel:.3f}s")

total = t_pre + t_sel
sel_pct = 100 * t_sel / total
pre_pct = 100 * t_pre / total
print(f"  Ratio: precompute={pre_pct:.0f}%  selection={sel_pct:.0f}%")
print(f"  Diag: evaluated={diag['candidates_evaluated']} "
      f"pruned={diag['candidates_pruned']} "
      f"fully_evaluated={diag.get('candidates_fully_evaluated', '?')} "
      f"hits={diag['signal_cache_hits']} misses={diag['signal_cache_misses']}")

print()
print("  >>> INTERPRETATION <<<")
if sel_pct > 50:
    print("  >>> Selection loop DOMINATES.")
    print("  >>> Stage 3 (parallel precompute) alone has a ceiling.")
    print("  >>> Compiled kernels (Stage 5) help BOTH precompute and the")
    print("  >>> per-scenario simulations inside the selection loop — bigger win.")
elif sel_pct > 25:
    print("  >>> Precompute and selection roughly balanced.")
    print("  >>> Stage 5 (compiled kernels) wins on both fronts since the")
    print("  >>> selection loop's per-scenario work uses the same feature kernels.")
else:
    print("  >>> Precompute dominates (good — Stage 3 parallelism is the right")
    print("  >>> target, Stage 5 amplifies it).")

# Per-scenario breakdown to help estimate how many feature-kernel invocations
# happen inside the selection loop.
n_cand = diag['candidates_evaluated']
# Baseline sim + up to len(scenarios) stressed sims per candidate (minus prunes)
max_sims = n_cand * (1 + len(scenarios))
actual_sims = max_sims - diag['candidates_pruned'] * len(scenarios) // 2  # rough estimate
print()
print(f"  Back-of-envelope: selection ran up to {max_sims} simulations "
      f"(baseline + stress) for {n_cand} candidates × {len(scenarios)} scenarios.")
print(f"  Per-simulation time: ~{t_sel/max(max_sims,1)*1000:.1f}ms avg "
      f"(includes signal reuse from cache + run_simulation + scoring)")

SECTION 7 (retry) — MR Selection loop vs precompute wall-time ratio
  Built 6 full candidates (with engine_config)
  Serial precompute: 81.502s  (13.584s per candidate)
  Stress scenarios loaded: 17
  Selection loop:    0.350s
  Ratio: precompute=100%  selection=0%
  Diag: evaluated=6 pruned=4 fully_evaluated=2 hits=6 misses=0

  >>> INTERPRETATION <<<
  >>> Precompute dominates (good — Stage 3 parallelism is the right
  >>> target, Stage 5 amplifies it).

  Back-of-envelope: selection ran up to 108 simulations (baseline + stress) for 6 candidates × 17 scenarios.
  Per-simulation time: ~3.2ms avg (includes signal reuse from cache + run_simulation + scoring)
